In [ ]:
import sys
import numpy as np
import pinocchio as pin
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

sys.path.append('./python/bsqp')
sys.path.append('./python')

from bsqp.mpc_controller import MPC_GATO
from bsqp.common import figure8
from bsqp.config import (
    FIG8_DEFAULT_PARAMS,
    TIAGO_RIGHT_START_CONFIGS,
    BATCH_COLORS,
)

np.random.seed(42)

print("Imports complete")

In [ ]:
config = {
    'batch_sizes': [1],
    'N': 32,
    'dt': 0.01,
    'sim_time': 16.0,
    'sim_dt': 0.001,
    'start_config': 'comfortable',
    'f_ext': np.zeros(6),
    'solver_params': {
        'max_sqp_iters': 20,
        'kkt_tol': 1e-3,
        'max_pcg_iters': 120,
        'pcg_tol': 1e-3,
        'solve_ratio': 1.0,
        'mu': 1.0,
        'q_cost': 160.0,
        'qd_cost': 1e-2,
        'u_cost': 1e-5,
        'N_cost': 50.0,
        'q_lim_cost': 0.01,
        'vel_lim_cost': 0.0,
        'ctrl_lim_cost': 0.0,
        'rho': 0.01,
    },
}

print("Configuration:")
print(f"  Batch sizes: {config['batch_sizes']}")
print(f"  Horizon: N={config['N']}, dt={config['dt']}s")
print(f"  Simulation: {config['sim_time']}s at {1/config['sim_dt']:.0f}Hz")
print(f"  External force: {config['f_ext'][:3]} N")

In [ ]:
# Load robot model
urdf_path = "../gato/dynamics/tiago_right/tiago_right_arm.urdf"
model = pin.buildModelFromUrdf(urdf_path)

print(f"  Loaded model")
print(f"  Joints: {model.njoints-1}")
print(f"  DOF: {model.nq}")
print(f"  Generalized velocity: {model.nv}")

In [ ]:
# Reference EE trajectory
x_start_q = TIAGO_RIGHT_START_CONFIGS[config['start_config']]
data = model.createData()
pin.forwardKinematics(model, data, x_start_q)
pin.updateFramePlacements(model, data)
torso_id = model.getFrameId("torso_lift_link")
tool_id = model.getFrameId("arm_right_tool_link")
center = (data.oMf[torso_id].inverse() * data.oMf[tool_id]).translation

fig8_params = {
    **FIG8_DEFAULT_PARAMS,
    'A_x': 0.04,
    'A_z': 0.04,
    'offset': [float(center[0]), float(center[1]), float(center[2] - 0.02)],
    'theta': 0.0,
}
fig8_traj = figure8(config['dt'], **fig8_params)

# Visualize
ref_points = fig8_traj.reshape(-1, 6)[:, :3]
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot(ref_points[:, 0], ref_points[:, 1], ref_points[:, 2], 'r-', alpha=0.6)
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_zlabel('Z [m]')
ax.view_init(elev=20, azim=135)
plt.show()

print(f"  Total points: {len(ref_points)}")
print(f"  Duration: {len(ref_points) * config['dt']:.1f}s")
print(f"  Center: {center}")

In [ ]:
results = {}

# starting configuration
x_start = np.hstack((TIAGO_RIGHT_START_CONFIGS[config['start_config']], np.zeros(model.nv)))

print("Running experiments...\n")
print("=" * 60)

for batch_size in config['batch_sizes']:
    print(f"\nBatch size {batch_size}:")
    print("-" * 40)

    # Create controller
    mpc = MPC_GATO(
        model=model,
        model_path=urdf_path,
        N=config['N'],
        dt=config['dt'],
        batch_size=batch_size,
        constant_f_ext=config['f_ext'],
        track_full_stats=True,
        plant_type='tiago_right',
        solver_params=config['solver_params'],
        reset_dual_each_tick=True,
        warm_start_policy='repeat_current',
    )

    _, stats = mpc.run_mpc_fig8(
        x_start=x_start,
        fig8_traj=fig8_traj,
        sim_dt=config['sim_dt'],
        sim_time=config['sim_time']
    )

    results[batch_size] = stats
    print(f"Completed {len(stats['timestamps'])} iterations")

print("\n" + "=" * 60)
print("All experiments complete")

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4), squeeze=False)
axes = axes[0]
batch_sizes = sorted(results.keys())

for idx, batch_size in enumerate(batch_sizes):
    ax = axes[idx]

    # Plot reference traj (dotted line)
    ax.plot(ref_points[:, 0], ref_points[:, 2], ':',
            linewidth=1.0, alpha=0.5, label='Reference')

    # Plot actual trajectory
    ee_actual = results[batch_size]['ee_actual']
    color = BATCH_COLORS.get(batch_size, '#000000')
    ax.plot(ee_actual[:, 0], ee_actual[:, 2],
            color=color, linewidth=1.5, label=f'Batch Size = {batch_size}', alpha=0.8)

    ax.set_ylabel('Z [m]', fontsize=14)
    ax.set_xlabel('X [m]', fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.axis('equal')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=2)

plt.tight_layout()
plt.subplots_adjust(top=0.97)
plt.show()